In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import os
import yfinance as yf

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping



warnings.simplefilter(action="ignore", category=FutureWarning)

In [2]:
import os
import random

def fijar_semillas(semilla=42):
    # 1. Fijar semilla de Python
    os.environ['PYTHONHASHSEED'] = str(semilla)
    random.seed(semilla)
    
    # 2. Fijar semilla de NumPy
    np.random.seed(semilla)
    
    # 3. Fijar semilla de TensorFlow/Keras
    tf.random.set_seed(semilla)
    
    print(f"[*] Semillas fijadas a {semilla} para asegurar reproducibilidad.")

# Llamar a la función antes de crear ningún modelo ni dividir datos
fijar_semillas(42)

[*] Semillas fijadas a 42 para asegurar reproducibilidad.


In [3]:
# =====================================================================
# 1. DESCARGA Y PREPARACIÓN DE DATOS (Código del Profesor)
# =====================================================================
print("Descargando datos de Yahoo Finance...")
start_date = '1960-01-01'
tickers_validos = ['AEP', 'BA', 'CAT', 'CNP', 'CVX', 'DIS', 'DTE', 'ED', 'GD', 'GE', 
                   'HON', 'HPQ', 'IBM', 'IP', 'JNJ', 'KO', 'KR', 'MMM', 'MO', 'MRK', 
                   'MSI', 'PG', 'XOM']

precios_close = yf.download(tickers_validos, start=start_date, auto_adjust=True, progress=False)['Close']
precios_close.dropna(axis=1, inplace=True)

# Cálculo de retornos logarítmicos
returns = np.log(precios_close).diff().dropna()
print(f"Forma de los datos de retornos: {returns.shape}")

# Función del profesor para crear ventanas
def create_time_series_data(data, input_window_size, output_window_size):
    X, y = [], []
    data_array = data.values if isinstance(data, pd.DataFrame) else data
    num_features = data_array.shape[1] 

    for i in range(len(data_array) - input_window_size - output_window_size + 1):
        input_sequence = data_array[i : i + input_window_size]
        X.append(input_sequence)
        
        if output_window_size > 0:
            output_sequence = data_array[i + input_window_size : i + input_window_size + output_window_size]
            average_output = np.mean(output_sequence, axis=0) 
            y.append(average_output)
        else:
            y.append(data_array[i + input_window_size - 1])
            
    return np.array(X), np.array(y)

Descargando datos de Yahoo Finance...


c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:144: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:144: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a f

Forma de los datos de retornos: (16190, 23)


In [10]:
# =====================================================================
# 2. DEFINICIÓN DE ARQUITECTURA Y BASELINES
# =====================================================================

def construir_modelo_rnn(config, input_shape, n_assets=23):
    """Construye un modelo dinámico basado en la configuración dada."""
    model = Sequential()
    model.add(Input(shape=input_shape))
    
    # Capa dinámica (LSTM o GRU)
    CapaRecurrente = config['tipo_capa']
    model.add(CapaRecurrente(config['neuronas'], return_sequences=False))
    
    model.add(Dropout(config['dropout']))
    model.add(Dense(n_assets)) # Salida: 23 valores (promedio de los 23 activos)
    
    optimizador = Adam(learning_rate=config['lr'])
    model.compile(optimizer=optimizador, loss='mae')
    return model

def calcular_baselines(X_test, y_test, y_train_mean):
    """Calcula el MAE para modelos simples, incluyendo Buy and Hold."""
    
    # 1. Baseline Naive: El futuro será igual al último día de la ventana de entrada
    y_pred_naive = X_test[:, -1, :]
    mae_naive = np.mean(np.abs(y_pred_naive - y_test))
    
    # 2. Baseline SMA: El futuro será igual a la media de la ventana de entrada actual
    y_pred_sma = np.mean(X_test, axis=1)
    mae_sma = np.mean(np.abs(y_pred_sma - y_test))
    
    # 3. Baseline Buy and Hold: Predecir siempre la media histórica del entrenamiento
    # Creamos un array del mismo tamaño que y_test relleno con la media de y_train
    y_pred_bh = np.full_like(y_test, y_train_mean)
    mae_bh = np.mean(np.abs(y_pred_bh - y_test))
    
    return mae_naive, mae_sma, mae_bh

# Crear carpeta para guardar gráficas si no existe
os.makedirs('graficas_convergencia', exist_ok=True)

In [12]:
# =====================================================================
# 3. CONFIGURACIÓN DEL EXPERIMENTO
# =====================================================================

input_windows = [5, 10, 30, 90]
output_windows = [1, 5, 30, 90]

# 1. Lista para ventanas con POCA información (In: 5 y 10)
# Definimos el punto de partida común para clonarlo fácilmente
base_configs = [
    {'tipo_capa': GRU,  'neuronas': 32, 'dropout': 0.0, 'lr': 0.001},
    {'tipo_capa': LSTM, 'neuronas': 64, 'dropout': 0.0, 'lr': 0.001}
]

# Inicializamos las 8 listas independientes. 
# Usamos list() para que sean copias independientes y puedas modificarlas en el futuro
hp_in5_corto  = [
    # Ganador IN 5 OUT 1  |||| IN 5 OUT 5  
    # 1. El Campeón Defensor (Lo mantenemos como control para comparar)
    {'tipo_capa': GRU,  'neuronas': 8, 'dropout': 0.0, 'lr': 0.001},
    
    # 2. El Micro-Cerebro Extremo: Si 8 neuronas se atascan, probamos con 4.
    # Obligamos a la red a ser una simple calculadora de tendencias muy básica.
    {'tipo_capa': GRU,  'neuronas': 4, 'dropout': 0.0, 'lr': 0.001},
    
    # 3. El Cambio de Familia: LSTM diminuta.
    # A veces, la forma en la que la LSTM maneja sus puertas internas filtra 
    # mejor el ruido a cortísimo plazo que la GRU.
    {'tipo_capa': LSTM, 'neuronas': 4, 'dropout': 0.0, 'lr': 0.001},
    
    # 4. El "Contrariano": Subimos neuronas pero metemos Dropout.
    # Le damos 16 neuronas para que intente ver algo más complejo, pero le 
    # apagamos el 10% (Dropout 0.1) para que no pueda memorizar el ruido.
    {'tipo_capa': GRU,  'neuronas': 16, 'dropout': 0.1, 'lr': 0.001}

]
'''
la diferencia entre tu modelo (0.010628) y el Buy & Hold (0.010571) es de 0.000057. 
¡Es un empate técnico absoluto! Básicamente, el modelo se da cuenta de que es imposible predecir el ruido de mañana 
con los 5 días anteriores y hace exactamente lo mismo que el Buy & Hold. Si tras ejecutar esta lista sigues a 0.00005 puntos del Buy & Hold,
debes detenerte y aceptarlo como el resultado definitivo
'''



hp_in5_largo  = [

    # Ganador IN 5 OUT 30 
    # 1. El Campeón Defensor (Control)
    {'tipo_capa': GRU,  'neuronas': 32, 'dropout': 0.1, 'lr': 0.0005},

    # 2. El Minimalista a Medio Plazo: Bajamos a 8 neuronas. Si 5 días 
    # no dan para mucho, no necesitamos 32 neuronas para procesarlos.
    {'tipo_capa': GRU,  'neuronas': 8,  'dropout': 0.0, 'lr': 0.0005},

    # Ganador IN 5 OUT 90 
    # 3. El Campeón Defensor (Control)
    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.3, 'lr': 0.0005},

    # 4. El Cambio de Motor: Probamos una GRU ligera en lugar del "Titán" LSTM.
    # Al tener menos puertas lógicas, la GRU a veces promedia mejor a largo plazo.
    {'tipo_capa': GRU,  'neuronas': 16,  'dropout': 0.1, 'lr': 0.0005}

]
'''
Si el Buy & Hold sigue ganando por la mínima, tendremos la evidencia definitiva (y documentada en tus gráficas)
de que el problema no es el algoritmo, sino la estacionariedad de los datos de entrada
'''


hp_in10_corto = [
    
    # Ganador IN 10 OUT 1 ||||| IN 10 OUT 5
    #nuevo gaador AMBOS
    {'tipo_capa': LSTM, 'neuronas': 16, 'dropout': 0.2, 'lr': 0.0005},

    # 1. La Micro-GRU: Cambiamos a GRU (suele funcionar mejor con poco ruido)
    # y bajamos a 4 neuronas sin Dropout. Máxima simplicidad.
    {'tipo_capa': GRU,  'neuronas': 4, 'dropout': 0.0, 'lr': 0.001},
    
    # 2. LSTM Minimalista: Mantenemos la familia LSTM pero le cortamos la 
    # memoria a la mitad (8 neuronas) y le quitamos el Dropout.
    {'tipo_capa': LSTM, 'neuronas': 8, 'dropout': 0.0, 'lr': 0.0005},
    
    # 3. El "Caminante Ligero": GRU de 8 neuronas pero con pasos más cortos.
    # Buscamos que se ajuste lentamente a los 10 días sin saltar a la media.
    {'tipo_capa': GRU,  'neuronas': 8, 'dropout': 0.0, 'lr': 0.0005}

]


hp_in10_largo = [

    # Ganador IN 10 OUT 30 ||| Ganador IN 10 OUT 90 
    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.25, 'lr': 0.00005},

    # 1. El "Termómetro" Ligero: GRU de 16 neuronas.
    # Mucho más rápida y con menos parámetros que la LSTM pesada. 
    # LR moderado y sin Dropout para que vea los datos sin filtros.
    {'tipo_capa': GRU,  'neuronas': 16, 'dropout': 0.0, 'lr': 0.0005},
    
    # 2. La LSTM Desatada: 32 neuronas, LR estándar (0.001) y SIN Dropout.
    # Vamos a quitarle todos los frenos. Queremos ver si al dejarla correr 
    # es capaz de encontrar algún patrón, o si directamente hace Overfitting.
    {'tipo_capa': LSTM, 'neuronas': 32, 'dropout': 0.0, 'lr': 0.001},
    
    # 3. El Micro-Cerebro para Largo Plazo: Solo 8 neuronas.
    # Si 10 días solo contienen una única señal de tendencia (alcista/bajista), 
    # 8 neuronas son más que suficientes para capturarla sin confundirse con el ruido.
    {'tipo_capa': LSTM,  'neuronas': 8, 'dropout': 0.0, 'lr': 0.0005}
    
]

hp_in30_corto = [
    # Ganador IN 30 OUT 1 ||| Ganador IN 30 OUT 5
    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.3, 'lr': 0.00005},

    # Al tener 30 días de entrada, 8 neuronas actuarán como un embudo estricto: 
    # solo la señal más fuerte pasará a la predicción. Sin Dropout, LR estándar.
    {'tipo_capa': GRU,  'neuronas': 8,  'dropout': 0.0, 'lr': 0.001},
    
    # 2. La LSTM Minimalista: 16 neuronas.
    # Algo más de memoria por si los 30 días esconden un patrón cíclico mensual,
    # pero sin pasarnos de capacidad para no sobreajustar al ruido.
    {'tipo_capa': LSTM, 'neuronas': 16, 'dropout': 0.0, 'lr': 0.0005},
    
    # 3. El Micro-Cerebro Extremo: GRU de 4 neuronas.
    # Obligamos al modelo a resumir un mes entero de bolsa en solo 4 números.
    # Si la tendencia principal no aparece aquí, es que no existe.
    {'tipo_capa': GRU,  'neuronas': 4,  'dropout': 0.0, 'lr': 0.001}

]


hp_in30_largo = [
    # Ganador IN 30 OUT 30  |||| Ganador IN 30 OUT 90 
    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.3, 'lr': 0.0005},

    # 1. El "Radar" Ligero: GRU de 32 neuronas SIN Dropout.
    # Suficiente capacidad para procesar un mes de datos, pero sin filtros 
    # que le borren la información. LR moderado para que baje suave.
    {'tipo_capa': GRU,  'neuronas': 32, 'dropout': 0.0, 'lr': 0.0005},
    
    # 2. El Micro-Cerebro a Largo Plazo: LSTM de 16 neuronas.
    # Si la relación entre el mes pasado y el próximo trimestre es simple, 
    # 16 neuronas la encontrarán sin perderse en detalles irrelevantes.
    {'tipo_capa': LSTM, 'neuronas': 16, 'dropout': 0.0, 'lr': 0.0005},
    
    # 3. La LSTM Desatada: 64 neuronas, LR estándar y SIN Dropout.
    # Le quitamos todos los frenos. Queremos ver si una red de tamaño medio, 
    # sin restricciones de regularización, es capaz de superar la media 
    # o si se estrella contra el Overfitting.
    {'tipo_capa': LSTM, 'neuronas': 64, 'dropout': 0.0, 'lr': 0.001}
]
    


hp_in90_corto = [
    # Ganador IN 90 OUT 1 
    {'tipo_capa': LSTM, 'neuronas': 16, 'dropout': 0.0, 'lr': 0.001},

    # Ganador IN 90 OUT 5
    {'tipo_capa': LSTM, 'neuronas': 16, 'dropout': 0.1, 'lr': 0.001},

    # 1. El Embudo Extremo: GRU de 8 neuronas.
    # Usamos GRU porque es computacionalmente más ágil para secuencias muy largas (90 pasos).
    # Obligamos a la red a comprimir 90 días de ruido en solo 8 números.
    {'tipo_capa': GRU,  'neuronas': 8,  'dropout': 0.0, 'lr': 0.001},
    
    # 2. La Micro-LSTM: 8 neuronas, sin filtros (sin Dropout).
    # Si la tendencia macroeconómica del trimestre afecta al día de mañana, 
    # esto es lo mínimo necesario para capturarla sin sobreajustar.
    {'tipo_capa': LSTM, 'neuronas': 8, 'dropout': 0.0, 'lr': 0.001},
    
    # 3. La GRU Lenta y Observadora: 16 neuronas, LR más bajo (0.0005).
    # Le damos un poco más de "cerebro" para procesar los 3 meses, 
    # pero a pasos pequeños para evitar que entre en pánico y salte a la media plana en la época 1.
    {'tipo_capa': GRU,  'neuronas': 16, 'dropout': 0.0, 'lr': 0.0005}

]

hp_in90_largo = [
    # Ganador IN 90 OUT 30 
    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.2, 'lr': 0.00005},

    # Ganador IN 90 OUT 90
    {'tipo_capa': LSTM, 'neuronas': 64, 'dropout': 0.1, 'lr': 0.0005},

    # 1. El "Radar Macro" Equilibrado: GRU de 32 neuronas SIN Dropout.
    # La GRU es ideal aquí: al tener menos puertas matemáticas que la LSTM, 
    # procesa la larga secuencia de 90 días de forma más eficiente y con 
    # menos riesgo de memorizar basura.
    {'tipo_capa': GRU,  'neuronas': 32, 'dropout': 0.0, 'lr': 0.0005},
    
    # 2. El Micro-Cerebro Trimestral: LSTM de 16 neuronas.
    # Capacidad minúscula para una ventana de 90 días. Si hay un patrón 
    # estacional o de ciclo de mercado, 16 neuronas lo verán. Si no, 
    # al menos no sobreajustará.
    {'tipo_capa': LSTM, 'neuronas': 16, 'dropout': 0.0, 'lr': 0.0005},
    
    # 3. La GRU Ultra-Ligera con filtro: 16 neuronas y un toque de Dropout (0.1).
    # Máxima compresión de información con un pequeñísimo filtro para el ruido.
    {'tipo_capa': GRU,  'neuronas': 16, 'dropout': 0.1, 'lr': 0.0005}
    
    
]

lista_hiperparametros = []

# Matrices para reportar resultados finales de las Redes Recurrentes
matriz_mae_train_rnn = np.zeros((4, 4))
matriz_mae_val_rnn = np.zeros((4, 4))
matriz_mae_rnn = np.zeros((4, 4)) # Esta es la de Test que ya tenías

matriz_mae_naive = np.zeros((4, 4))
matriz_mae_sma = np.zeros((4, 4))
matriz_mae_bh = np.zeros((4, 4))

# Aumentamos la paciencia a 15 épocas
early_stop = EarlyStopping(
    monitor ='val_loss', 
    patience = 20,               # <--- CAMBIO AQUÍ
    restore_best_weights = True  # IMPORTANTE: Que devuelva los pesos de la mejor época
)


In [13]:
# =====================================================================
# 4. BUCLE PRINCIPAL (AUTOMATIZACIÓN DE LOS 16 MODELOS x CONFIGURACIONES)
# =====================================================================


# =====================================================================
# RECORDATORIO: Inicializa estas nuevas matrices antes del bucle
# =====================================================================
matriz_mae_naive_val = np.zeros((len(input_windows), len(output_windows)))
matriz_mae_sma_val = np.zeros((len(input_windows), len(output_windows)))
matriz_mae_bh_val = np.zeros((len(input_windows), len(output_windows)))

print("\nIniciando entrenamiento de modelos...")

for i, in_w in enumerate(input_windows):
    for j, out_w in enumerate(output_windows):
        print(f"\n=======================================================")
        print(f" Ventana Entrada: {in_w} días | Ventana Salida: {out_w} días")
        print(f"=======================================================")
        
        # 1. Crear datos
        X, y = create_time_series_data(returns, in_w, out_w)
        
        # 2. Separación CRONOLÓGICA: 80% Train, 10% Validacion, 10% Test
        # split_1 = int(len(X) * 0.8)
        # split_2 = int(len(X) * 0.9)
        
        # Para un esquema 70% Train, 20% Validacion, 10% Test
        split_1 = int(len(X) * 0.70) # Aquí cortamos el Train
        split_2 = int(len(X) * 0.90) # Aquí cortamos la Validación (del 70% al 90% = 20%)
        
        X_train, y_train = X[:split_1], y[:split_1]
        X_val, y_val = X[split_1:split_2], y[split_1:split_2]
        X_test, y_test = X[split_2:], y[split_2:]

        # Calculamos la media global de entrenamiento para esta ventana (Buy and Hold)
        y_train_mean = np.mean(y_train, axis=0)
        
        
        # =====================================================================
        # 3. Baselines (AHORA EN VALIDACIÓN Y TEST)
        # =====================================================================

        '''
        mae_naive, mae_sma, mae_bh = calcular_baselines(X_test, y_test, y_train_mean)
        matriz_mae_naive[i, j] = mae_naive
        matriz_mae_sma[i, j] = mae_sma
        matriz_mae_bh[i, j] = mae_bh

        print(f"Baseline Naive (MAE en Test): {mae_naive:.6f}")
        print(f"Baseline SMA   (MAE en Test): {mae_sma:.6f}")
        print(f"Baseline Buy & Hold (MAE): {mae_bh:.6f}")
        '''

        # Calcular en Validación
        mae_naive_val, mae_sma_val, mae_bh_val = calcular_baselines(X_val, y_val, y_train_mean)
        matriz_mae_naive_val[i, j] = mae_naive_val
        matriz_mae_sma_val[i, j] = mae_sma_val
        matriz_mae_bh_val[i, j] = mae_bh_val
        
        # Calcular en Test
        mae_naive_test, mae_sma_test, mae_bh_test = calcular_baselines(X_test, y_test, y_train_mean)
        matriz_mae_naive[i, j] = mae_naive_test
        matriz_mae_sma[i, j] = mae_sma_test
        matriz_mae_bh[i, j] = mae_bh_test
        

        print("--- Baselines VALIDACIÓN ---")
        print(f"Naive: {mae_naive_val:.6f} | SMA: {mae_sma_val:.6f} | Buy&Hold: {mae_bh_val:.6f}")
        print("--- Baselines TEST ---")
        print(f"Naive: {mae_naive_test:.6f} | SMA: {mae_sma_test:.6f} | Buy&Hold: {mae_bh_test:.6f}\n")


        # 4. Búsqueda del mejor modelo recurrente
        mejor_val_loss = float('inf')
        mejor_modelo = None
        mejor_historial = None
        mejor_config = None


        # DIVIDE Y VENCERÁS: Selección de hiperparámetros
        if in_w == 5:
            lista_a_probar = hp_in5_corto if out_w in [1, 5] else hp_in5_largo
            nombre_lista = "In:5 Corto" if out_w in [1, 5] else "In:5 Largo"
            
        elif in_w == 10:
            lista_a_probar = hp_in10_corto if out_w in [1, 5] else hp_in10_largo
            nombre_lista = "In:10 Corto" if out_w in [1, 5] else "In:10 Largo"
            
        elif in_w == 30:
            lista_a_probar = hp_in30_corto if out_w in [1, 5] else hp_in30_largo
            nombre_lista = "In:30 Corto" if out_w in [1, 5] else "In:30 Largo"
            
        elif in_w == 90:
            lista_a_probar = hp_in90_corto if out_w in [1, 5] else hp_in90_largo
            nombre_lista = "In:90 Corto" if out_w in [1, 5] else "In:90 Largo"

        print(f" -> Usando banco de pruebas: [{nombre_lista}]")
        
        for config in lista_a_probar:
            capa_nombre = config['tipo_capa'].__name__
            print(f" -> Entrenando: {capa_nombre}, Neuronas: {config['neuronas']}, LR: {config['lr']}, DropOut: {config['dropout']}")
            
            modelo = construir_modelo_rnn(config, input_shape=(in_w, 23))
            
            # Usamos verbose=0 para no llenar la pantalla de números, epochs=50 es suficiente con EarlyStop
            historial = modelo.fit(X_train, y_train, 
                                   validation_data=(X_val, y_val),
                                   epochs=50, 
                                   batch_size=64, 
                                   callbacks=[early_stop], 
                                   verbose=0
                                   )
            
            val_loss_actual = min(historial.history['val_loss'])
            
            if val_loss_actual < mejor_val_loss:
                mejor_val_loss = val_loss_actual
                mejor_modelo = modelo
                mejor_historial = historial
                mejor_config = config
        
        print(f"\n[GANADOR] {mejor_config['tipo_capa'].__name__} ({mejor_config['neuronas']} neuronas)")
        
        # 5. Evaluación final del GANADOR en TRAIN, VALIDACIÓN y TEST
        mae_train_ganador = mejor_modelo.evaluate(X_train, y_train, verbose=0)
        mae_val_ganador = mejor_modelo.evaluate(X_val, y_val, verbose=0)
        mae_test_ganador = mejor_modelo.evaluate(X_test, y_test, verbose=0)
        
        # Guardar en sus respectivas matrices
        matriz_mae_train_rnn[i, j] = mae_train_ganador
        matriz_mae_val_rnn[i, j] = mae_val_ganador
        matriz_mae_rnn[i, j] = mae_test_ganador
        
        print(f"MAE del Modelo Ganador en TRAIN:      {mae_train_ganador:.6f}")
        print(f"MAE del Modelo Ganador en VALIDACIÓN: {mae_val_ganador:.6f}")
        print(f"MAE del Modelo Ganador en TEST:       {mae_test_ganador:.6f}")
        
        # 6. Guardar Gráfica de Convergencia del Ganador
        plt.figure(figsize=(10, 5))
        plt.plot(mejor_historial.history['loss'], label='Error Entrenamiento (MAE)')
        plt.plot(mejor_historial.history['val_loss'], label='Error Validación (MAE)')
        
        # Título con todos los hiperparámetros
        nombre_capa = mejor_config['tipo_capa'].__name__
        n_neuronas = mejor_config['neuronas']
        l_rate = mejor_config['lr']
        d_out = mejor_config['dropout']
        
        plt.title(f"Convergencia {nombre_capa} | Neuronas: {n_neuronas} | LR: {l_rate} | Drop: {d_out}\n(Ventana In:{in_w} - Out:{out_w})")
        
        plt.xlabel('Épocas')
        plt.ylabel('MAE')
        plt.legend()
        plt.grid(True)
        
        # Guardar la imagen
        nombre_archivo = f"graficas_convergencia/conver_in{in_w}_out{out_w}.png"
        plt.savefig(nombre_archivo)
        plt.close()


Iniciando entrenamiento de modelos...

 Ventana Entrada: 5 días | Ventana Salida: 1 días
--- Baselines VALIDACIÓN ---
Naive: 0.015405 | SMA: 0.011786 | Buy&Hold: 0.010571
--- Baselines TEST ---
Naive: 0.017789 | SMA: 0.013608 | Buy&Hold: 0.012243

 -> Usando banco de pruebas: [In:5 Corto]
 -> Entrenando: GRU, Neuronas: 8, LR: 0.001, DropOut: 0.0
 -> Entrenando: GRU, Neuronas: 4, LR: 0.001, DropOut: 0.0
 -> Entrenando: LSTM, Neuronas: 4, LR: 0.001, DropOut: 0.0
 -> Entrenando: GRU, Neuronas: 16, LR: 0.001, DropOut: 0.1

[GANADOR] LSTM (4 neuronas)
MAE del Modelo Ganador en TRAIN:      0.011873
MAE del Modelo Ganador en VALIDACIÓN: 0.010601
MAE del Modelo Ganador en TEST:       0.012265

 Ventana Entrada: 5 días | Ventana Salida: 5 días
--- Baselines VALIDACIÓN ---
Naive: 0.011802 | SMA: 0.006919 | Buy&Hold: 0.004730
--- Baselines TEST ---
Naive: 0.013656 | SMA: 0.008029 | Buy&Hold: 0.005580

 -> Usando banco de pruebas: [In:5 Corto]
 -> Entrenando: GRU, Neuronas: 8, LR: 0.001, DropOut:

In [7]:
# =====================================================================
# 5. RESULTADOS FINALES (Tablas para tu GitHub y Presentación)
# =====================================================================

print("\n\n" + "="*50)
print("MATRIZ DE RESULTADOS FINALES EN ENTRENAMIENTO (RNN)")
print("="*50)
df_rnn_train = pd.DataFrame(matriz_mae_train_rnn, 
                            index=[f'In_{w}' for w in input_windows], 
                            columns=[f'Out_{w}' for w in output_windows])
print(df_rnn_train)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS FINALES EN VALIDACIÓN (RNN)")
print("="*50)
df_rnn_val = pd.DataFrame(matriz_mae_val_rnn, 
                          index=[f'In_{w}' for w in input_windows], 
                          columns=[f'Out_{w}' for w in output_windows])
print(df_rnn_val)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS FINALES EN TEST (RNN)")
print("="*50)
df_rnn = pd.DataFrame(matriz_mae_rnn, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_rnn)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE NAIVE (TEST)")
print("="*50)
df_naive = pd.DataFrame(matriz_mae_naive, 
                        index=[f'In_{w}' for w in input_windows], 
                        columns=[f'Out_{w}' for w in output_windows])
print(df_naive)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE NAIVE (VALIDACION)")
print("="*50)
df_naive_val = pd.DataFrame(matriz_mae_naive_val, 
                        index=[f'In_{w}' for w in input_windows], 
                        columns=[f'Out_{w}' for w in output_windows])
print(df_naive_val)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE SMA (TEST)")
print("="*50)
df_sma = pd.DataFrame(matriz_mae_sma, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_sma)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE SMA (VALIDACION)")
print("="*50)
df_sma_val = pd.DataFrame(matriz_mae_sma_val, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_sma_val)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE BUY AND HOLD (TEST)")
print("="*50)
df_bh = pd.DataFrame(matriz_mae_bh, 
                     index=[f'In_{w}' for w in input_windows], 
                     columns=[f'Out_{w}' for w in output_windows])
print(df_bh)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE BUY AND HOLD (TEST)")
print("="*50)
df_bh_val = pd.DataFrame(matriz_mae_bh_val, 
                     index=[f'In_{w}' for w in input_windows], 
                     columns=[f'Out_{w}' for w in output_windows])
print(df_bh_val)



MATRIZ DE RESULTADOS FINALES EN ENTRENAMIENTO (RNN)
          Out_1     Out_5    Out_30    Out_90
In_5   0.011892  0.005528  0.002206  0.001273
In_10  0.011998  0.005738  0.002522  0.001700
In_30  0.011971  0.005873  0.002231  0.001286
In_90  0.011949  0.005632  0.002529  0.001329

MATRIZ DE RESULTADOS FINALES EN VALIDACIÓN (RNN)
          Out_1     Out_5    Out_30    Out_90
In_5   0.010628  0.004772  0.001943  0.001121
In_10  0.010689  0.004918  0.002201  0.001455
In_30  0.010673  0.004994  0.001996  0.001163
In_90  0.010684  0.004862  0.002226  0.001199

MATRIZ DE RESULTADOS FINALES EN TEST (RNN)
          Out_1     Out_5    Out_30    Out_90
In_5   0.012311  0.005614  0.002332  0.001275
In_10  0.012355  0.005785  0.002647  0.001743
In_30  0.012386  0.005925  0.002380  0.001303
In_90  0.012341  0.005747  0.002655  0.001373

MATRIZ DE RESULTADOS BASELINE NAIVE (TEST)
          Out_1     Out_5    Out_30    Out_90
In_5   0.017789  0.013656  0.012518  0.012254
In_10  0.017790  0.013657 